<a href="https://colab.research.google.com/github/Barazan19/hf-llm-course-journey/blob/main/ch01_transformer_models_1_4_how_do_transformers_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Pembahasan Lengkap: How do Transformers work?

Bagian ini merangkum materi fundamental dari kursus Hugging Face Bab 1 mengenai arsitektur Transformer, sejarahnya, jenis-jenis model, hingga perbedaan mendasar antara arsitektur dan checkpoint.

---

## 1. A Bit of Transformer History (Sejarah Singkat)

* **Kapan dimulai?** Arsitektur Transformer diperkenalkan pada **Desember 2017** lewat paper legendaris *"Attention Is All You Need"* oleh tim peneliti Google.
* **Fokus Utama:** Sebelum Transformer, model NLP menggunakan RNN (Recurrent Neural Networks) seperti LSTM atau GRU. Masalah utama RNN adalah lambat karena membaca kata secara berurutan (sekuensial). Transformer merevolusi ini dengan melakukan komputasi secara paralel memanfaatkan **Self-Attention Mechanism**.
* **Timeline Perkembangan Utama:**
  * **Juni 2018 (GPT):** Model pertama bertipe *Decoder-only* yang digunakan untuk *fine-tuning* berbagai tugas NLP.
  * **Oktober 2018 (BERT):** Model bertipe *Encoder-only* yang jago dalam memahami konteks kalimat secara dua arah (bi-directional).
  * **2019 - Sekarang (GPT-2, T5, BART, GPT-3, GPT-4, Llama):** Model semakin membesar (miliaran parameter) dan kemampuan generatifnya semakin luar biasa.
```

```markdown
## 2. Transformers are Language Models (Tipe-tipe Model Bahasa)

Model bahasa (*Language Models*) dilatih secara mandiri (*self-supervised training*) menggunakan data teks yang sangat besar tanpa perlu label manual dari manusia.

Secara umum ada 3 kategori utama model Transformer:
1. **Encoder-only (Auto-encoding models):** Jago memahami kalimat secara utuh. (Contoh: BERT, RoBERTa)
2. **Decoder-only (Auto-regressive models):** Jago menghasilkan teks baru kata demi kata. (Contoh: GPT-2, GPT-3, Llama)
3. **Encoder-Decoder (Sequence-to-sequence models):** Jago menerjemahkan teks atau membuat ringkasan. (Contoh: T5, BART)
```

```markdown
## 3. Transformers are Big Models & Transfer Learning

### Kenapa model Transformer sangat besar?
Model seperti GPT atau BERT memiliki ratusan juta hingga ratusan miliar parameter. Untuk melatih model sebesar ini dari nol (*from scratch*) membutuhkan biaya jutaan dolar dan komputer super cepat.

### Konsep Utama: Transfer Learning
Alih-alih melatih model dari nol, kita menggunakan **Transfer Learning**:
1. **Pretraining (Latihan Awal):** Model dilatih pada dataset raksasa (misal seluruh isi internet) untuk memahami struktur bahasa dasar. Ini sangat mahal.
2. **Fine-Tuning (Penyesuaian):** Kita mengambil model yang sudah di-*pretrain* tersebut, lalu melatihnya sedikit menggunakan dataset kita sendiri yang jauh lebih kecil untuk tugas spesifik (misal klasifikasi sentimen). Ini sangat murah dan cepat!

Mari kita contohkan proses **Fine-Tuning** secara sederhana (inferensi menggunakan pre-trained model untuk klasifikasi):
```

In [5]:
# Contoh Transfer Learning: Menggunakan Pipeline untuk Sentimen Analisis
# Kita menggunakan model yang sudah di-pretrain dan langsung dipakai (Zero-shot/Inferensi)
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
hasil = classifier("I absolutely love learning about Transformer models!")
print(hasil)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998106360435486}]


```markdown
## 4. General Transformer Architecture & The Original Architecture

Arsitektur asli Transformer terdiri dari dua blok utama:
* **Encoder (Kiri):** Menerima input teks dan mengubahnya menjadi representasi numerik (vektor) berdimensi tinggi yang kaya akan makna konteks kalimat.
* **Decoder (Kanan):** Menerima representasi numerik dari Encoder serta teks yang sudah dihasilkan sebelumnya untuk memprediksi kata berikutnya.

Berikut adalah ilustrasi kode sederhana untuk membedakan output dari komponen Encoder vs Decoder:
```

In [6]:
# Mari kita muat model Encoder-only (BERT) untuk melihat representasi numerik (embeddings) yang dihasilkannya
from transformers import AutoModel, AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

text = "Transformers are revolutionary."
inputs = bert_tokenizer(text, return_tensors="pt")
outputs = bert_model(**inputs)

# Output dari Encoder adalah 'last_hidden_state' (vektor representasi makna)
print("Bentuk representasi output Encoder (BERT) [batch_size, sequence_length, hidden_dim]:")
print(outputs.last_hidden_state.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bentuk representasi output Encoder (BERT) [batch_size, sequence_length, hidden_dim]:
torch.Size([1, 6, 768])


```markdown
## 5. Attention Layers

Kunci kehebatan Transformer terletak pada **Attention Layers** (khususnya *Self-Attention*).

### Apa itu Attention?
Attention secara harfiah artinya **"Perhatian"**. Mekanisme ini memungkinkan model untuk memberikan bobot atau perhatian lebih pada kata-kata tertentu dalam sebuah kalimat ketika memproses kata yang sedang aktif.

**Contoh Klasik:**
* *"The animal didn't cross the street because **it** was too tired."*
* *"The animal didn't cross the street because **it** was too wide."*

Pada kalimat pertama, kata **"it"** mengarah ke **"animal"**.
Pada kalimat kedua, kata **"it"** mengarah ke **"street"**.

Dengan *Attention*, komputer tahu kata **"it"** itu harus dipasangkan ke kata yang mana berdasarkan kata di sekitarnya.
```

```markdown
## 6. Architectures vs. Checkpoints

Dalam dunia Hugging Face, sangat penting untuk membedakan dua istilah ini:
* **Architecture (Arsitektur):** Cetak biru atau struktur kode dari model (misalnya: struktur jaringan neural GPT-2, BERT, T5). Arsitektur ini seperti resep makanan kosong.
* **Checkpoint / Weights (Bobot):** Nilai parameter (angka-angka) aktual yang didapat setelah proses latihan (*training*). Checkpoint ini seperti masakan yang sudah jadi.

**Analogi Gampang:**
* *Architecture:* Kerangka mobil (misal: Honda Civic).
* *Checkpoint:* Mobil fisik yang sudah jadi dengan warna dan spesifikasi tertentu (misal: Honda Civic berwarna Merah keluaran tahun 2023).
```